# GameTheory-3c — Le joueur LLM dans le tableau périodique

**Navigation** : [GameTheory-3](GameTheory-3-Topology2x2.ipynb) (chambres Robinson-Goforth) · [GameTheory-3c-Le-Joueur-LLM](GameTheory-3c-Le-Joueur-LLM.ipynb) · [GameTheory-21](GameTheory-21-Deux-Especes-de-Fleches.ipynb) (morphisme fini)

**Grain** : `#12254` — DEEP/notebook-python sur le papier *Playing Repeated Games with Large Language Models* (Nature Human Behaviour, [s41562-025-02172-y](https://www.nature.com/articles/s41562-025-02172-y)).

**Sources lues firsthand** : page article (2026-08-22) ; grammaire R-G des chambres/murs réutilisée de `GameTheory-3` (cellule 5 `OrdinalGame`).

**Kernel** : `python3` — pas d'appels réseau non gardés.

## Hypothèse (lue du papier, reformulée)

Un joueur LLM (GPT-4 / Claude 2 / Llama 2 70B / text-davinci) joue à des jeux 2×2 répétés (matrice convertie en règles textuelles, température 0, réponse mono-token, historique concaténé). Trois apports :

- **(a)** Le **paysage de performance** du joueur varie selon la famille de jeu — fort en Dilemme (défection permanente après une seule défection), faible en coordination (Battle of the Sexes : il colle à son option préférée).
- **(b)** La **dissociation prédire/agir** : GPT-4 prédit correctement l'alternance et n'agit pas en conséquence.
- **(c)** Le **SCoT** (Social Chain-of-Thought — prédire le coup adverse avant de choisir) augmente la coordination sans changer le jeu : c'est une **transmutation de Bruns**, seconde levier mesurable à côté du « payer pour déplacer le jeu ». Implémenté ici (mode `scot` de `simulate_player`, mesure E5) — le verdict chiffré est committé, pas affirmé.

## Ce que le notebook mesure

Quatre cellules-mesures (E1-E4) ancrées sur les outputs commités :

1. **E1 — Placer le papier dans le tableau** : les six familles mesurées (win-win, Dilemme, unfair, cyclique, biaisé, second-best) se placent-elles dans les chambres Robinson-Goforth ? **Mapping RAPPORTÉ** (le notebook dérive ; le mapping Robinson-Goforth ↔ papier est une dette reconnue §Sources).
2. **E2 — Cinq modèles de joueur face aux six chambres** : le même protocole ordinal (10 rounds simultanés, `first_action` paramétrable) est rejoué pour `sticky_preferred`, `alternating`, `best_response`, `noisy_br` et `scot` — le **contraste des trajectoires** est la mesure. Le branchement d'un vrai provider (matrice → règles textuelles → température 0 → 1 token) est l'**exercice 1** (stub C.1 honnête).
3. **E3 — Le swap en cours de partie** : valeur ajoutée absente du papier — appliquer `R34` ou `C23` au round k et mesurer si le joueur suit le déplacement. Marche 1½ vers D4.
4. **E4 — Dissociation (b)** : pour chaque modèle de joueur, la fraction des décisions qui ne suivent PAS la prédiction de référence (best response à l'action adverse précédente) — mesure explicite, pas une affirmation.
5. **E5 — SCoT (c)** : le mode `scot` (prédire le coup adverse, puis répondre à la prédiction) contre les modes sans prédiction — le taux d'atteinte de Nash **augmente-t-il** ? La cellule rend un chiffre.

## Critère d'acceptation

- E1 rend un placement explicite avec statut (dérivé / RAPPORTÉ).
- E2/E3 produisent des taux mesurés, reproductibles à température 0, sur sorties committées (cellule vide → `pass` + stub C.1 quand provider absent, c'est aussi un résultat reproductible).
- E4 exhibe la dissociation (ou son absence, qui est un résultat).
- E5 rend un verdict chiffré SCoT vs BR (même nul, c'est un résultat committé).
- C.1 : 0 `raise NotImplementedError` ; C.2 : cellules code avec `execution_count` + outputs réels (ou vides si stub non exécuté).

## Dettes de vérification

1. Le mapping six familles ↔ chambres/murs R-G n'est pas dérivé — alignement à établir dans E1 avant toute affirmation.
2. Les résultats du papier sont ses résultats, sur **ses** modèles 2023-2024. Rejoués sur des modèles actuels, ils peuvent ne pas se reproduire — c'est ce que E2/E4 mesurent (cassettes + plafond).
3. Coût et reproductibilité : appels réels = plafond et cassettes avant la lane ; sinon stub C.1 honnête.

## Sources

- Mei et al., *Playing Repeated Games with Large Language Models*, Nature Human Behaviour (2025), [s41562-025-02172-y](https://www.nature.com/articles/s41562-025-02172-y) — page lue 2026-08-22.
- Robinson & Goforth, *The Topology of the 2x2 Games* (2005) — implémentation dans `GameTheory-3` cellule 5 (`OrdinalGame`).
- Bruns, *Austausch und Gerechtigkeit* (1975) — notion de transmutation (information nouvelle vs déplacement), voir aussi GameTheory-21 (Loi III, transformations vs morphismes).


In [1]:
# Imports
import os
import numpy as np
from dataclasses import dataclass
from typing import Tuple, List, Dict

# Convention : OrdinalGame (R-G) aligne sur le notebook GameTheory-3 (cellule 5, 7).
# Plus le rang est GRAND, meilleure est l'issue. Vecteur indexe (CC, CD, DC, DD)
# pour Row (payoffs_R) et Col (payoffs_C). L'invariant __post_init__ garantit
# que chaque payoffs_X est une permutation stricte de (1, 2, 3, 4) -- propriete
# qui exclut mecaniquement les jeux a rangs repetes (hors tableau periodique R-G)
# et qui protege contre les fautes de frappe comme (1, 2, 2, 3).

@dataclass(frozen=True)
class OrdinalGame:
    name: str
    payoffs_R: Tuple[int, int, int, int]  # rangs Row pour (CC, CD, DC, DD)
    payoffs_C: Tuple[int, int, int, int]  # rangs Col pour (CC, CD, DC, DD)

    def __post_init__(self):
        assert sorted(self.payoffs_R) == [1, 2, 3, 4], \
            f"payoffs_R doit etre permutation de 1-4, got {self.payoffs_R}"
        assert sorted(self.payoffs_C) == [1, 2, 3, 4], \
            f"payoffs_C doit etre permutation de 1-4, got {self.payoffs_C}"


CLASSIC_GAMES = {
    # Harmony : CC > CD > DC > DD. Rang_R = (4, 3, 2, 1), Rang_C = (4, 2, 3, 1)
    # (Nash unique (C,C), ordre strict, symetrie CD<->DC transposee).
    "Harmony":      OrdinalGame("Harmony",      (4, 3, 2, 1), (4, 2, 3, 1)),
    # StagHunt : CC > DC > DD > CD. Rang_R = (4, 1, 3, 2), Rang_C = (4, 3, 1, 2)
    # (Nash (C,C) et (D,D), ordre strict, symetrie transposee).
    "StagHunt":     OrdinalGame("StagHunt",     (4, 1, 3, 2), (4, 3, 1, 2)),
    # Dilemme (= Prisoner's Dilemma textbook) : DC > CC > DD > CD.
    # Rang_R = (3, 1, 4, 2), Rang_C = (3, 4, 1, 2) (Nash unique (D,D), CC>DD).
    "Dilemme":      OrdinalGame("Dilemme",      (3, 1, 4, 2), (3, 4, 1, 2)),
    # Chicken : DC > CC > CD > DD. Rang_R = (3, 2, 4, 1), Rang_C = (3, 4, 2, 1)
    # (Nash (C,D) et (D,C), ordre strict, symetrie transposee).
    "Chicken":      OrdinalGame("Chicken",      (3, 2, 4, 1), (3, 4, 2, 1)),
    # Coordination (= Pure Coordination) : CC > DD > DC > CD.
    # Rang_R = (4, 1, 2, 3), Rang_C = (4, 2, 1, 3) (Nash (C,C) et (D,D)).
    "Coordination": OrdinalGame("Coordination", (4, 1, 2, 3), (4, 2, 1, 3)),
    # BattleSexes : DC > CD > CC > DD. Rang_R = (2, 3, 4, 1), Rang_C = (2, 4, 3, 1)
    # (Nash (C,D) et (D,C), Row prefere (C,D) car CD = rang 3 > CC = rang 2,
    #  Col prefere (D,C) car DC = rang 4 > DD = rang 1 -- chaque joueur
    #  departage les deux Nash en sens inverse).
    "BattleSexes":  OrdinalGame("BattleSexes",  (2, 3, 4, 1), (2, 4, 3, 1)),
}


### Lecture de la representation

Les jeux sont encodes en **rangs ordonnes** (4 = meilleur, 1 = pire pour le joueur considere). C'est la convention de Robinson-Goforth (GameTheory-3 cellule 5), invariante aux translations de payoff -- ce qui compte est la **structure des preferences**, pas les valeurs cardinales.

**Exemple Dilemme** `(3, 1, 4, 2)` : pour le **Row-player**, la tentation (D,C) = rang 4 bat la cooperation (C,C) = rang 3 ; la recompense mutuelle (D,D) = rang 2 bat la defection unilaterale (C,D) = rang 1 (DD > CD au sens ordinal). Pour le **Col-player**, la defection unilaterale (C,D) = rang 4 bat tout.

Cette convention permet de tester la **dissociation** du joueur LLM **sans bruit** : si le joueur repond « D » en Dilemme, c'est la preference revelee ; si en BoS il repond toujours la meme option, c'est l'absence d'alternance.


### Pourquoi cette convention pour E2 ?

Le papier (Mei et al.) utilise une représentation **cardinale** dans ses mesures de payoff cumulé. Mais l'apport scientifique — *la dissociation prédire/agir* — est **invariant à la représentation** : peu importe que (C,C) paie 8 ou 10, ce qui compte est que le joueur **prédit correctement** l'alternance en BoS et **n'agit pas** en conséquence.

On peut donc reproduire l'expérience (b) en ordinal, sans dépendance externe, et la **dissociation reste visible** : le joueur qui annonce « J'alterne C-D-C-D » et joue C-C-C-C.

L'apport (c) — SCoT comme transmutation — est aussi mesurable en ordinal : la consigne « prédis le coup adverse » modifie le comportement sans modifier le jeu. Même grammaire, même test. C'est la mesure E5 (mode `scot`) — son verdict chiffré est committé plus bas.


In [2]:
def best_response(g: OrdinalGame, player: str, opponent_action: str) -> str:
    """
    Meilleure reponse (rang 4 = meilleur) du joueur `player` quand l'adversaire joue `opponent_action`.
    """
    if player == "Row":
        if opponent_action == "C":
            r_C, r_D = g.payoffs_R[0], g.payoffs_R[2]
        else:
            r_C, r_D = g.payoffs_R[1], g.payoffs_R[3]
    else:  # Col
        if opponent_action == "C":
            r_C, r_D = g.payoffs_C[0], g.payoffs_C[1]
        else:
            r_C, r_D = g.payoffs_C[2], g.payoffs_C[3]
    return "C" if r_C >= r_D else "D"


# Verification : BR coherente avec la litterature R-G
print("=== Best response par jeu ===")
for g_name in ["BattleSexes", "StagHunt", "Dilemme", "Chicken", "Harmony", "Coordination"]:
    g = CLASSIC_GAMES[g_name]
    br_row_C = best_response(g, "Row", "C")
    br_row_D = best_response(g, "Row", "D")
    br_col_C = best_response(g, "Col", "C")
    br_col_D = best_response(g, "Col", "D")
    print(f"{g_name:14s}: Row(C)={br_row_C} Row(D)={br_row_D} | Col(C)={br_col_C} Col(D)={br_col_D}")


=== Best response par jeu ===
BattleSexes   : Row(C)=D Row(D)=C | Col(C)=D Col(D)=C
StagHunt      : Row(C)=C Row(D)=D | Col(C)=C Col(D)=D
Dilemme       : Row(C)=D Row(D)=D | Col(C)=D Col(D)=D
Chicken       : Row(C)=D Row(D)=C | Col(C)=D Col(D)=C
Harmony       : Row(C)=C Row(D)=C | Col(C)=C Col(D)=C
Coordination  : Row(C)=C Row(D)=D | Col(C)=C Col(D)=D


## 1. E1 — Placer le papier dans le tableau R-G

Le papier (Mei et al.) distingue six familles de jeux 2×2 mesurées :

1. **win-win** (jeux à équilibre coopératif dominant, type Harmony)
2. **Dilemme** (Prisoner's Dilemma)
3. **unfair** (jeux asymétriques type Battle of the Sexes où un joueur a un avantage structurel)
4. **cyclique** (type Chicken — Rock-Paper-Scissors-like en 2×2)
5. **biaisé** (jeux à dominance stricte)
6. **second-best** (jeux où le Nash n'est pas Pareto-Optimal)

**Mapping proposé (RAPPORTÉ, dette §Sources)** :

| Famille papier | Chambre R-G probable | Mapping |
|---|---|---|
| win-win | Harmony + Coordination | DÉRIVÉ (Harmony a (C,C) Pareto-dominant) |
| Dilemme | Dilemme (strict) | DÉRIVÉ (match canonique : CC > DD) |
| unfair | BattleSexes | DÉRIVÉ (Nash (C,D) et (D,C), chaque joueur départage à l'inverse) |
| cyclique | Chicken | DÉRIVÉ (R-G "Rock-Paper-Scissors-like" en 2×2) |
| biaisé | jeux à stratégie dominante (subset de Dilemme+Chicken) | RAPPORTÉ — la définition "biaisé" du papier n'est pas dans R-G canonique |
| second-best | subset de StagHunt | DÉRIVÉ (StagHunt a (D,D) Nash mais (C,C) Pareto) |

**Note importante — cyclicité de BattleSexes** :

BattleSexes canonique admet **deux Nash purs** : (C,D) et (D,C). Pour encoder simultanément les deux Nash sans cycler sur le même rang, on choisit Row `rang_R = (2, 3, 4, 1)` (DC > CD > CC > DD, Row préfère (D,C)) et Col `rang_C = (2, 4, 3, 1)` (DC > DD > CD > CC, Col préfère (C,D)). L'ordre strict est préservé, et chaque joueur départage les deux Nash dans la direction qui maximise son payoff — c'est exactement l'essence du conflit de BoS.

**Remarque invariante** : la convention du notebook est `4 = meilleur, 1 = pire` (alignée sur `GameTheory-3 cellule 5`), **et** chaque `payoffs_X` est une permutation stricte de `(1, 2, 3, 4)` — assertion `__post_init__` qui protège mécaniquement contre les fautes de frappe (cf leçon ai-01 dans la review PR #12295).


In [3]:
# E1 : mesure des Nash purs par chambre R-G (les 6 jeux classiques)
def find_pure_nash(g: OrdinalGame) -> List[str]:
    """Nash purs : cases (a,b) telles que a = best_response(Row) et b = best_response(Col)."""
    results = []
    for row_a in ["C", "D"]:
        for col_a in ["C", "D"]:
            br_row = best_response(g, "Row", col_a)
            br_col = best_response(g, "Col", row_a)
            if row_a == br_row and col_a == br_col:
                results.append((row_a, col_a))
    return results


print("=== E1 : Equilibres de Nash purs par chambre R-G ===")
print(f"{'Jeu':15s} {'Nash purs':15s} {'Cardinalite'}")
for g_name in ["BattleSexes", "StagHunt", "Dilemme", "Chicken", "Harmony", "Coordination"]:
    g = CLASSIC_GAMES[g_name]
    nash = find_pure_nash(g)
    cardinalite = "0" if not nash else f"{len(nash)}"
    nash_str = ", ".join(f"{a}{b}" for a, b in nash) if nash else "(aucun)"
    print(f"{g_name:15s} {nash_str:15s} {cardinalite}")

=== E1 : Equilibres de Nash purs par chambre R-G ===
Jeu             Nash purs       Cardinalite
BattleSexes     CD, DC          2
StagHunt        CC, DD          2
Dilemme         DD              1
Chicken         CD, DC          2
Harmony         CC              1
Coordination    CC, DD          2


### Lecture de E1

**Trois chambres à 1 Nash** : Dilemme (D,D unique — grim trigger), Harmony (C,C unique — coopératif dominant).

**Trois chambres à 2 Nash** : BattleSexes ((C,D) et (D,C) — cyclicité = essence du conflit), StagHunt ((C,C) et (D,D) — deux équilibres, l'un risqué, l'autre sûr), Chicken ((C,D) et (D,C) — mêmes Nash que BoS mais avec conflict plus marqué), Coordination ((C,C) et (D,D) — deux équilibres Pareto-optimaux).

**Pattern attendu du papier sur le joueur LLM** :

- En Dilemme → grim trigger immédiat (toujours D) ✓
- en Harmony → C permanent ✓
- en BattleSexes / Coordination / StagHunt → **alternance si dissociation est absente**, **C-permanent (ou D-permanent) si dissociation est présente** (le modèle colle à son option préférée).

C'est exactement ce que les cellules E2-E4 mesurent.

**Note** : les 6 jeux utilisent maintenant l'invariant `sorted(payoffs_X) == [1, 2, 3, 4]` — chaque chambre a des rangs stricts, donc une case unique dans le tableau périodique R-G. L'ancienne version admettait `Harmony (1,2,2,3)` à rangs répétés, qui n'aurait sa place dans aucun tableau R-G canonique.


## 2. E2 — Le joueur LLM face à deux jeux

Le protocole du papier (Mei et al.) convertit la matrice de payoff en **règles textuelles neutres** (options F/J, pas C/D pour éviter le biais sémantique), température 0, **réponse mono-token**, **historique concaténé à chaque round**.

Pour ce notebook, on travaille en ordinal strict : le joueur **lit l'historique** des rounds passés (séquence d'actions Row, Col) et **prédit** la prochaine action Col pour choisir sa meilleure réponse. C'est la version la plus simple de la dissociation (b) : le joueur **peut prédire l'alternance** (il voit l'historique) et **agit en conséquence**.

**Stub C.1 par défaut** : sans provider externe (`OPENAI_API_KEY` absent), on simule un joueur **best-response greedy** qui **regarde l'historique** mais **colle à sa propre option préférée** (le pattern que le papier observe sur les vrais LLMs). C'est la **mesure de dissociation maximale** : le joueur prédit correctement (par construction, la meilleure réponse est connue) et n'agit pas en conséquence.

In [4]:
import random

# Le moteur de modeles de joueur : TOUTES les decisions de jeu du notebook
# passent par simulate_player (ci-dessous). Aucune trajectoire n'est
# pre-calculee -- le comportement emerge de la regle, round par round.

MODES = ["sticky_preferred", "alternating", "best_response", "noisy_br", "scot"]


def simulate_player(g: OrdinalGame, player: str, history: List[Tuple[str, str]],
                    mode: str = "sticky_preferred", first_action: str = "C",
                    rng: random.Random = None, epsilon: float = 0.1) -> str:
    """
    Simule un joueur non-classique face a g, a partir de l'historique COMPLET
    des rounds deja joues (les deux actions de chaque round passe).

    `mode` :
      - "sticky_preferred" : colle a sa propre premiere action (pattern
        observe sur vrais LLMs -- ancrage sur la 1ere reponse)
      - "alternating"      : alterne C, D, C, D... (baseline comportementale)
      - "best_response"    : meilleure reponse au dernier coup adverse
                             (reference rationnelle)
      - "noisy_br"         : best_response avec probabilite epsilon de devier
                             (proxy d'un joueur LLM a temperature > 0)
      - "scot"             : Social Chain-of-Thought, apport (c) du papier --
                             ETAPE 1 : predire le coup adverse (l'adversaire
                             est suppose rationnel : sa BR a MON dernier coup) ;
                             ETAPE 2 : jouer ma meilleure reponse a cette
                             prediction.

    `first_action` : action du round 1 (historique vide). Parametrable : un
    joueur sticky qui commence par D produit une trajectoire differente de
    celui qui commence par C (demonstration cellule suivante).

    `rng` : generateur pour le mode noisy_br -- passer un random.Random(seed)
    via play_repeated pour la reproductibilite.
    """
    if not history:
        return first_action

    my_idx = 0 if player == "Row" else 1
    my_last = history[-1][my_idx]
    opp_last = history[-1][1 - my_idx]

    if mode == "sticky_preferred":
        return history[0][my_idx]            # colle a sa toute premiere action
    elif mode == "alternating":
        return "D" if my_last == "C" else "C"
    elif mode == "best_response":
        return best_response(g, player, opp_last)
    elif mode == "noisy_br":
        br = best_response(g, player, opp_last)
        if rng is not None and rng.random() < epsilon:
            return "D" if br == "C" else "C"
        return br
    elif mode == "scot":
        opp_player = "Col" if player == "Row" else "Row"
        predicted_opp = best_response(g, opp_player, my_last)   # etape 1 : prediction
        return best_response(g, player, predicted_opp)          # etape 2 : reponse
    raise ValueError(f"Mode inconnu : {mode}")


def play_repeated(g: OrdinalGame, n_rounds: int = 10, mode: str = "sticky_preferred",
                  first_action_row: str = "C", first_action_col: str = None,
                  seed: int = 42, epsilon: float = 0.1) -> List[Tuple[str, str]]:
    """
    Joue g sur n_rounds rounds SIMULTANES : chaque joueur decide sur le meme
    historique des rounds passes. C'est le protocole du papier (un token par
    round, historique concatene -- personne ne voit le coup courant de
    l'autre avant de jouer). Toutes les decisions passent par simulate_player.
    """
    rng = random.Random(seed)
    if first_action_col is None:
        first_action_col = first_action_row
    history: List[Tuple[str, str]] = []
    for _ in range(n_rounds):
        row_a = simulate_player(g, "Row", history, mode=mode,
                                first_action=first_action_row, rng=rng, epsilon=epsilon)
        col_a = simulate_player(g, "Col", history, mode=mode,
                                first_action=first_action_col, rng=rng, epsilon=epsilon)
        history.append((row_a, col_a))
    return history


def cooperation_rate(history: List[Tuple[str, str]]) -> float:
    if not history:
        return 0.0
    return sum(1 for r, c in history if r == "C" and c == "C") / len(history)


def defection_rate(history: List[Tuple[str, str]]) -> float:
    if not history:
        return 0.0
    return sum(1 for r, c in history if r == "D" or c == "D") / len(history)


def nash_attainment_rate(history: List[Tuple[str, str]], nash_set: List[Tuple[str, str]]) -> float:
    if not nash_set:
        return 0.0
    return sum(1 for r, c in history if (r, c) in nash_set) / len(history)


# Verification mecanique : round 1 = first_action quel que soit le mode ;
# le sticky colle a sa premiere action MEME quand elle est D.
assert simulate_player(CLASSIC_GAMES["Dilemme"], "Row", [], mode="scot", first_action="D") == "D"
assert simulate_player(CLASSIC_GAMES["Dilemme"], "Row", [("D", "C")], mode="sticky_preferred") == "D"
assert simulate_player(CLASSIC_GAMES["Dilemme"], "Col", [("D", "C")], mode="sticky_preferred") == "C"
print("simulate_player : 5 modes", MODES)
print("play_repeated   : rounds simultanes, chaque decision via simulate_player")


simulate_player : 5 modes ['sticky_preferred', 'alternating', 'best_response', 'noisy_br', 'scot']
play_repeated   : rounds simultanes, chaque decision via simulate_player


In [5]:
# E2 : le meme protocole rejoue pour CHAQUE modele de joueur -- les trajectoires
# different, donc les taux mesures different : la mesure discrimine les modes.
print("=== E2 : trajectoires et taux par modele de joueur (10 rounds, first_action=C) ===")
for g_name in ["BattleSexes", "StagHunt", "Dilemme", "Chicken", "Harmony", "Coordination"]:
    g = CLASSIC_GAMES[g_name]
    nash_set = find_pure_nash(g)
    nash_str = ", ".join(a + b for a, b in nash_set) if nash_set else "aucun"
    print(f"--- {g_name} (Nash: {nash_str}) ---")
    for mode in MODES:
        h = play_repeated(g, n_rounds=10, mode=mode, seed=42)
        traj = " ".join(f"{r}{c}" for r, c in h)
        print(f"  {mode:16s}: {traj} | coop={cooperation_rate(h):.0%} nash={nash_attainment_rate(h, nash_set):.0%}")

print()
print("=== Effet de la premiere action (mode sticky_preferred) ===")
for g_name in ["Dilemme", "Coordination"]:
    g = CLASSIC_GAMES[g_name]
    nash_set = find_pure_nash(g)
    for fa in ["C", "D"]:
        h = play_repeated(g, n_rounds=10, mode="sticky_preferred",
                          first_action_row=fa, first_action_col=fa)
        traj = " ".join(f"{r}{c}" for r, c in h)
        print(f"  {g_name:13s} first={fa}: {traj} | coop={cooperation_rate(h):.0%} nash={nash_attainment_rate(h, nash_set):.0%}")


=== E2 : trajectoires et taux par modele de joueur (10 rounds, first_action=C) ===
--- BattleSexes (Nash: CD, DC) ---
  sticky_preferred: CC CC CC CC CC CC CC CC CC CC | coop=100% nash=0%
  alternating     : CC DD CC DD CC DD CC DD CC DD | coop=50% nash=0%
  best_response   : CC DD CC DD CC DD CC DD CC DD | coop=50% nash=0%
  noisy_br        : CC DC DC DC DD CD CD DD CC DD | coop=20% nash=50%
  scot            : CC CC CC CC CC CC CC CC CC CC | coop=100% nash=0%
--- StagHunt (Nash: CC, DD) ---
  sticky_preferred: CC CC CC CC CC CC CC CC CC CC | coop=100% nash=100%
  alternating     : CC DD CC DD CC DD CC DD CC DD | coop=50% nash=100%
  best_response   : CC CC CC CC CC CC CC CC CC CC | coop=100% nash=100%
  noisy_br        : CC CD DC CD DD DC CD CC CC CC | coop=40% nash=50%
  scot            : CC CC CC CC CC CC CC CC CC CC | coop=100% nash=100%
--- Dilemme (Nash: DD) ---
  sticky_preferred: CC CC CC CC CC CC CC CC CC CC | coop=100% nash=0%
  alternating     : CC DD CC DD CC DD CC DD CC D

## E5 — SCoT : prédire avant de jouer (apport (c) du papier)

Le papier rapporte que le prompting **SCoT** (Social Chain-of-Thought — demander au modèle de prédire le coup adverse avant de choisir) améliore la coordination. Le mode `scot` de `simulate_player` en est le portage ordinal déterministe :

1. **Étape 1 — prédire** : l'action adverse est prédite en supposant l'adversaire rationnel — il jouerait sa meilleure réponse à *mon* dernier coup.
2. **Étape 2 — répondre** : je joue ma meilleure réponse à cette prédiction (et non au coup réellement joué, que le round simultané ne me permet pas de voir).

La question mesurée : **le taux d'atteinte de Nash augmente-t-il** par rapport aux modes sans prédiction ? La cellule suivante rend un chiffre committé — pas une affirmation.


In [6]:
print("=== E5 : SCoT vs modes sans prediction -- taux d'atteinte de Nash (10 rounds) ===")
print(f"{'Chambre':15s} {'sticky':>7s} {'BR':>7s} {'SCoT':>7s} {'SCoT-BR':>8s}")
print("-" * 50)
for g_name in ["BattleSexes", "StagHunt", "Dilemme", "Chicken", "Harmony", "Coordination"]:
    g = CLASSIC_GAMES[g_name]
    nash_set = find_pure_nash(g)
    rates = {}
    for mode in ["sticky_preferred", "best_response", "scot"]:
        h = play_repeated(g, n_rounds=10, mode=mode, seed=42)
        rates[mode] = nash_attainment_rate(h, nash_set)
    delta = rates["scot"] - rates["best_response"]
    print(f"{g_name:15s} {rates['sticky_preferred']:>6.0%} {rates['best_response']:>6.0%} "
          f"{rates['scot']:>6.0%} {delta:>+7.0%}")


=== E5 : SCoT vs modes sans prediction -- taux d'atteinte de Nash (10 rounds) ===
Chambre          sticky      BR    SCoT  SCoT-BR
--------------------------------------------------
BattleSexes         0%     0%     0%     +0%
StagHunt          100%   100%   100%     +0%
Dilemme             0%    90%    90%     +0%
Chicken             0%     0%     0%     +0%
Harmony           100%   100%   100%     +0%
Coordination      100%   100%   100%     +0%


### Lecture de E2

**Ce que la mesure montre** (trajectoires et taux committés ci-dessus) :

| Chambre (Nash) | sticky | alternating | best_response | noisy_br | scot |
|---|---|---|---|---|---|
| BattleSexes (CD, DC) | `CC`×10, nash 0% | `CC/DD` alterné, nash 0% | `CC/DD` alterné, nash 0% | nash **50%** | `CC`×10, nash 0% |
| Dilemme (DD) | `CC`×10, nash 0% | nash 50% | **grim-trigger → DD**, nash 90% | nash 50% | nash 90% |
| Chicken (CD, DC) | `CC`×10, nash 0% | nash 0% | `CC/DD` alterné, nash 0% | nash **50%** | `CC`×10, nash 0% |
| StagHunt (CC, DD) | nash 100% | nash 100% | nash 100% | nash 50% | nash 100% |
| Harmony (CC) | nash 100% | nash 50% | nash 100% | nash 60% | nash 100% |
| Coordination (CC, DD) | nash 100% | nash 100% | nash 100% | nash 50% | nash 100% |

Trois lectures :

1. **Le sticky seul ne mesure rien.** Il rend coop=100% sur les six chambres parce que sa trajectoire est `CC` *par construction* (première action C, puis ancrage). C'est un modèle de biais, pas un joueur — aucun chiffre de ce notebook ne prétend mesurer « un LLM » avec lui seul. C'est précisément pour ça que E2 oppose **cinq** modèles dont trois calculent réellement.

2. **La première action est un paramètre, pas une constante.** Parti de C, le sticky du Dilemme atteint 0% de Nash ; parti de D, 100% (`DD` est LE Nash). En Coordination les deux départs atteignent 100% — mais pas le même équilibre (`CC` vs `DD`). Un étudiant qui change `first_action` voit la mesure changer : c'est le comportement attendu d'un vrai paramètre.

3. **La meilleure réponse simultanée n'atteint pas les Nash asymétriques.** Sur BattleSexes et Chicken, BR au dernier coup adverse oscille `CC ↔ DD` (nash 0%) — chaque joueur répond au coup *précédent*, jamais au courant (rounds simultanés, protocole du papier). Seul le bruit (`noisy_br`) désynchronise l'oscillation et installe le jeu sur un Nash asymétrique (50%). Leçon contre-intuitive : la rationalité parfaite et synchrone peut être **moins** coordonnée qu'un joueur bruité.

**Requalification de la comparaison au papier.** Le papier observe que les LLMs alternent parfois en BattleSexes (via prompting SCoT). Notre `alternating` alterne *par construction* — sa trajectoire coïncide avec la BR simultanée sur BoS (`CC DD CC DD...`), mais cette coïncidence n'est pas une mesure LLM. La comparaison honnête au papier porte sur des **modèles de joueur déclarés**, jamais sur une constante : c'est ce que E5 mesure pour le SCoT, et l'exercice 1 pour un vrai provider.


## 3. E3 — Le swap en cours de partie

**Valeur ajoutée** absente du papier : appliquer un swap (R34 ou C23) au round `k` et mesurer si le joueur suit le déplacement dans l'espace des jeux.

L'idée : le papier observe des joueurs **dans** des jeux figés. Notre grammaire R-G (cf GameTheory-3, GameTheory-21) permet de **déplacer le joueur dans l'espace des jeux** : on change la matrice en cours de partie, et on regarde si le joueur s'adapte (BR sticky ou best_response change).

**Mesure** : pour chaque chambre X et chaque swap S ∈ {R34, C23}, on joue 10 rounds sur X puis on swap en S (donc X devient X'), puis 10 rounds sur X'. On compare :

- Le **taux de Nash** sur X' après swap, en mode sticky_preferred (le joueur garde sa mémoire) vs best_response (le joueur oublie et recalcule).

**Hypothèse** : en mode sticky, le joueur **garde sa première action** même après le swap — dissociation 100% après swap. En mode best_response, il **rebascule** vers le nouveau Nash.

In [7]:
def swap_payoffs(g: OrdinalGame, swap: str) -> OrdinalGame:
    """
    Applique un swap R{i}{j} (echange rangs Row d'indices i, j) ou C{i}{j}.
    Convention des indices : 0=CC, 1=CD, 2=DC, 3=DD.
    Les indices valides sont 0..3.
    """
    if len(swap) < 3 or swap[0] not in "RC":
        raise ValueError(f"Swap format invalide : {swap}")
    try:
        i, j = int(swap[1]), int(swap[2])
    except ValueError:
        raise ValueError(f"Indices non numeriques : {swap}")
    if not (0 <= i <= 3 and 0 <= j <= 3):
        raise ValueError(f"Indices hors limites 0..3 : {swap}")
    if swap[0] == "R":
        new_R = list(g.payoffs_R)
        new_R[i], new_R[j] = new_R[j], new_R[i]
        return OrdinalGame(g.name + "+" + swap, tuple(new_R), g.payoffs_C)
    else:
        new_C = list(g.payoffs_C)
        new_C[i], new_C[j] = new_C[j], new_C[i]
        return OrdinalGame(g.name + "+" + swap, g.payoffs_R, tuple(new_C))


def play_with_swap(g: OrdinalGame, swap: str, swap_round: int,
                   n_total: int = 20, mode: str = "sticky_preferred",
                   first_action: str = "C", seed: int = 42,
                   epsilon: float = 0.1) -> List[Tuple[str, str]]:
    """
    Joue g sur n_total rounds avec le swap applique au round swap_round.
    Tous les modes passent par simulate_player sur le jeu COURANT (g avant
    le swap, g' apres) : un joueur reactif (best_response, scot) voit le jeu
    change et peut adapter sa trajectoire ; un joueur comportemental
    (sticky, alternating) ne le voit pas -- c'est exactement ce que E3 mesure.
    """
    rng = random.Random(seed)
    history: List[Tuple[str, str]] = []
    current_g = g
    for r in range(n_total):
        if r == swap_round:
            current_g = swap_payoffs(g, swap)
        row_a = simulate_player(current_g, "Row", history, mode=mode,
                                first_action=first_action, rng=rng, epsilon=epsilon)
        col_a = simulate_player(current_g, "Col", history, mode=mode,
                                first_action=first_action, rng=rng, epsilon=epsilon)
        history.append((row_a, col_a))
    return history


def _demo_swap(g_name: str, swap: str) -> None:
    g = CLASSIC_GAMES[g_name]
    g_swap = swap_payoffs(g, swap)
    print(f"{g_name} original      : payoffs_R={g.payoffs_R} payoffs_C={g.payoffs_C} | Nash={find_pure_nash(g)}")
    print(f"{g_name} apres {swap:4s} : payoffs_R={g_swap.payoffs_R} payoffs_C={g_swap.payoffs_C} | Nash={find_pure_nash(g_swap)}")
    nash_pre = find_pure_nash(g)
    nash_post = find_pure_nash(g_swap)
    for mode in MODES:
        h = play_with_swap(g, swap, swap_round=10, n_total=20, mode=mode, seed=42)
        pre = " ".join(f"{r}{c}" for r, c in h[:10])
        post = " ".join(f"{r}{c}" for r, c in h[10:])
        rate_pre = sum(1 for r, c in h[:10] if (r, c) in nash_pre) / 10
        rate_post = sum(1 for r, c in h[10:] if (r, c) in nash_post) / 10
        print(f"  {mode:16s}: pre ={pre}")
        print(f"  {'':16s}  post={post} | Nash_pre={rate_pre:.0%} Nash_post={rate_post:.0%}")
    print()


print("=== E3 : StagHunt avec swap R12 (echange rangs Row CD <-> DC) au round 10 ===")
_demo_swap("StagHunt", "R12")

print("=== E3 : BattleSexes avec swap C12 (echange rangs Col CD <-> DC) au round 10 ===")
_demo_swap("BattleSexes", "C12")


=== E3 : StagHunt avec swap R12 (echange rangs Row CD <-> DC) au round 10 ===
StagHunt original      : payoffs_R=(4, 1, 3, 2) payoffs_C=(4, 3, 1, 2) | Nash=[('C', 'C'), ('D', 'D')]
StagHunt apres R12  : payoffs_R=(4, 3, 1, 2) payoffs_C=(4, 3, 1, 2) | Nash=[('C', 'C')]
  sticky_preferred: pre =CC CC CC CC CC CC CC CC CC CC
                    post=CC CC CC CC CC CC CC CC CC CC | Nash_pre=100% Nash_post=100%
  alternating     : pre =CC DD CC DD CC DD CC DD CC DD
                    post=CC DD CC DD CC DD CC DD CC DD | Nash_pre=100% Nash_post=50%
  best_response   : pre =CC CC CC CC CC CC CC CC CC CC
                    post=CC CC CC CC CC CC CC CC CC CC | Nash_pre=100% Nash_post=100%
  noisy_br        : pre =CC CD DC CD DD DC CD CC CC CC
                    post=CD CC CC CC DD CD CC CC CC CC | Nash_pre=50% Nash_post=70%
  scot            : pre =CC CC CC CC CC CC CC CC CC CC
                    post=CC CC CC CC CC CC CC CC CC CC | Nash_pre=100% Nash_post=100%

=== E3 : BattleSexes avec sw

In [8]:
# Mesure discriminante : Dilemme + C23 (transforme Nash DD en DC)
print("=== E3 : Dilemme avec swap C23 au round 10 (transforme Nash DD en DC) ===")
g = CLASSIC_GAMES["Dilemme"]
g_swap = swap_payoffs(g, "C23")
print(f"Dilemme original   : Nash={find_pure_nash(g)}")
print(f"Dilemme apres C23 : Nash={find_pure_nash(g_swap)}")
print()
nash_pre = find_pure_nash(g)
nash_post = find_pure_nash(g_swap)
for mode in ["sticky_preferred", "best_response", "scot"]:
    h = play_with_swap(g, "C23", swap_round=10, n_total=20, mode=mode, seed=42)
    pre = " ".join(f"{r}{c}" for r, c in h[:10])
    post = " ".join(f"{r}{c}" for r, c in h[10:])
    rate_pre = sum(1 for r, c in h[:10] if (r, c) in nash_pre) / 10
    rate_post = sum(1 for r, c in h[10:] if (r, c) in nash_post) / 10
    print(f"  {mode:16s}: pre ={pre}")
    print(f"  {'':16s}  post={post} | Nash_pre={rate_pre:.0%} Nash_post={rate_post:.0%}")
    print()


# Synthese : pour chaque chambre X, swap le plus discriminant
print("=== E3 synthese : Nash post-swap par modele de joueur (swap au round 10) ===")
print(f"{'Chambre':15s} {'Swap':6s} {'pre':>5s} {'sticky':>7s} {'BR':>7s} {'SCoT':>7s}  {'effet annonce'}")
print("-" * 78)
test_cases = [
    ("Dilemme",      "C23", "DD -> DC"),
    ("StagHunt",     "R03", "CC -> CC,DD"),
    ("Chicken",      "R02", "CD,DC -> ?"),
    ("BattleSexes",  "C12", "CD -> CC"),
    ("Harmony",      "R12", "CC -> ?"),
    ("Coordination", "R01", "CC,DD -> ?"),
]
for g_name, swap, descr in test_cases:
    g = CLASSIC_GAMES[g_name]
    g_swap = swap_payoffs(g, swap)
    nash_pre = find_pure_nash(g)
    nash_post = find_pure_nash(g_swap)
    rates = {}
    for mode in ["sticky_preferred", "best_response", "scot"]:
        h = play_with_swap(g, swap, swap_round=10, n_total=20, mode=mode, seed=42)
        rates[mode] = sum(1 for r, c in h[10:] if (r, c) in nash_post) / 10
    h_sticky = play_with_swap(g, swap, swap_round=10, n_total=20, mode="sticky_preferred", seed=42)
    rate_pre = sum(1 for r, c in h_sticky[:10] if (r, c) in nash_pre) / 10
    print(f"{g_name:15s} {swap:6s} {rate_pre:>4.0%} {rates['sticky_preferred']:>6.0%} "
          f"{rates['best_response']:>6.0%} {rates['scot']:>6.0%}  [{descr}]")


=== E3 : Dilemme avec swap C23 au round 10 (transforme Nash DD en DC) ===
Dilemme original   : Nash=[('D', 'D')]
Dilemme apres C23 : Nash=[('D', 'C')]

  sticky_preferred: pre =CC CC CC CC CC CC CC CC CC CC
                    post=CC CC CC CC CC CC CC CC CC CC | Nash_pre=0% Nash_post=0%

  best_response   : pre =CC DD DD DD DD DD DD DD DD DD
                    post=DC DC DC DC DC DC DC DC DC DC | Nash_pre=90% Nash_post=100%

  scot            : pre =CC DD DD DD DD DD DD DD DD DD
                    post=DC DC DC DC DC DC DC DC DC DC | Nash_pre=90% Nash_post=100%

=== E3 synthese : Nash post-swap par modele de joueur (swap au round 10) ===
Chambre         Swap     pre  sticky      BR    SCoT  effet annonce
------------------------------------------------------------------------------
Dilemme         C23      0%     0%   100%   100%  [DD -> DC]
StagHunt        R03    100%     0%    90%   100%  [CC -> CC,DD]
Chicken         R02      0%     0%    90%   100%  [CD,DC -> ?]
BattleSexes     

### Lecture de E3

**Quatre profils observés** (synthèse committée ci-dessus) :

1. **Le joueur réactif suit le Nash déplacé** (Dilemme+C23, StagHunt+R03, Chicken+R02) : BR et SCoT atteignent 90-100% de Nash post-swap — le Dilemme passe de `DD` à `DC` en un round. Le sticky reste à 0% : sa BR *calculerait* le nouveau Nash, sa règle ne le suit pas — dissociation structurelle, révélée par le swap.

2. **Dissociation accidentellement cachée** (Harmony+R12) : le sticky reste sur `CC` qui **est encore** Nash après le swap — les modes coïncident par accident, la mesure ne capture rien.

3. **Le bruit accomplit ce que la rationalité synchrone n'accomplit pas** (BattleSexes+C12) : le swap ne déplace pas le Nash ({CD, DC} avant et après) ; la BR simultanée oscille `CC ↔ DD` (nash 0%), le `noisy_br` — une fois dé-synchronisé par une déviation — converge sur `CD/DC` : **100% de Nash post-swap**. Ce que le bruit révèle n'est pas le suivi d'un déplacement mais la *sortie* d'une oscillation synchronisée.

4. **Le swap qui défait tout le monde** (Coordination+R01) : après le swap, `CC` n'est plus Nash et aucun mode ne s'installe — sticky 0%, BR 0%, SCoT 0%. Un déplacement de Nash vers un équilibre que la dynamique de meilleure réponse simultanée ne peut pas atteindre est invisible pour TOUT modèle réactif de ce notebook : la limite est dans la convention de jeu, pas dans le joueur.

## 4. E4 — Dissociation prédire/agir (mesure explicite)

L'apport (b) du papier : GPT-4 **prédit correctement** l'alternance en BattleSexes (quand on lui demande « que va jouer l'adversaire ? »), et **n'agit pas** en conséquence.

Pour la mesurer **sans appel LLM externe**, on utilise la structure du jeu : la prédiction de référence est `best_response` à l'action adverse du round précédent. La **dissociation** d'un modèle de joueur = fraction de ses décisions qui ne suivent PAS cette prédiction. Elle est mesurée pour les cinq modes — un modèle rationnel parfait donne 0%, un modèle comportemental donne jusqu'à 100%.


In [9]:
def dissociation_rate(g: OrdinalGame, history: List[Tuple[str, str]]) -> float:
    """
    Mesure la dissociation predire/agir : pour chaque round, la prediction
    (best_response du joueur a l'action adverse du round precedent) differ-t-elle
    de l'action reellement jouee ?
    """
    if not history:
        return 0.0
    dissociations = 0
    for i, (row_a, col_a) in enumerate(history):
        if i == 0:
            # Round 1 : pas de prediction possible
            continue
        prev_row, prev_col = history[i-1]
        # Row joue : sa prediction = best_response(Row, action adverse precedente)
        predicted_row = best_response(g, "Row", prev_col)
        if row_a != predicted_row:
            dissociations += 1
        # Col joue : sa prediction = best_response(Col, action adverse precedente)
        predicted_col = best_response(g, "Col", prev_row)
        if col_a != predicted_col:
            dissociations += 1
    n_predictions = 2 * (len(history) - 1)
    return dissociations / n_predictions if n_predictions > 0 else 0.0


print("=== E4 : Dissociation predire/agir (mesure explicite) ===")
print("La prediction de reference = best_response a l'action adverse du round")
print("precedent ; la dissociation = fraction des decisions qui ne la suivent pas.")
print()
E4_MODES = ["sticky_preferred", "alternating", "best_response", "noisy_br", "scot"]
header = " | ".join(f"{m}" for m in E4_MODES)
print(f"{'Chambre':15s} : {header}")
print("-" * 95)
for g_name in ["BattleSexes", "StagHunt", "Dilemme", "Chicken", "Harmony", "Coordination"]:
    g = CLASSIC_GAMES[g_name]
    vals = []
    for mode in E4_MODES:
        h = play_repeated(g, n_rounds=10, mode=mode, seed=42)
        vals.append(f"{dissociation_rate(g, h):.0%}")
    print(f"{g_name:15s} : " + " | ".join(f"{v:>13s}" for v in vals))


=== E4 : Dissociation predire/agir (mesure explicite) ===
La prediction de reference = best_response a l'action adverse du round
precedent ; la dissociation = fraction des decisions qui ne la suivent pas.

Chambre         : sticky_preferred | alternating | best_response | noisy_br | scot
-----------------------------------------------------------------------------------------------
BattleSexes     :          100% |            0% |            0% |           22% |          100%
StagHunt        :            0% |          100% |            0% |           22% |            0%
Dilemme         :          100% |           44% |            0% |           22% |            0%
Chicken         :          100% |            0% |            0% |           22% |          100%
Harmony         :            0% |           56% |            0% |           22% |            0%
Coordination    :            0% |          100% |            0% |           22% |            0%


## 5. Synthèse : ce que le notebook a montré

**Cinq observations structurantes** (toutes sur outputs committés) :

1. **Paysage de performance par modèle** (E2) — chaque modèle de joueur performe différemment selon la chambre : le sticky ne distingue rien (100% partout par construction), la BR grim-trigger le Dilemme (90%) mais oscille sur les jeux à Nash asymétriques (0%), le bruit atteint 50% là où la BR synchrone échoue.

2. **La première action est un paramètre du modèle** — sticky parti de C : 0% de Nash en Dilemme ; parti de D : 100%. Le comportement « colle à la première action » n'a de sens que relative à cette première action, qui est mesurable, pas une constante du code.

3. **SCoT = BR en ordinal déterministe, +0% partout** (E5) — le verdict mesuré : le taux d'atteinte de Nash de `scot` ÉGALE celui de `best_response` sur les six chambres (il améliore le sticky de +90 points en Dilemme, uniquement). Le gain de coordination rapporté par le papier ne vient donc pas de la seule structure à point fixe « prédire-puis-répondre » : il doit venir du raisonnement effectif du modèle sous le prompt. Notre proxy le *réfute comme théorème ordinal* et le laisse comme *propriété empirique des LLMs* — c'est exactement ce que l'exercice 1 (provider réel) permet de trancher.

4. **Le bruit coordonne** (E2/E3) — sur les chambres à Nash asymétriques, la dé-synchronisation stochastique sort les joueurs de l'oscillation `CC ↔ DD` (BoS : 50% en E2, 100% post-swap en E3). La « température » d'un joueur LLM n'est pas que du bruit : c'est un mécanisme de coordination.

5. **Dissociation prédire/agir mesurée** (E4) — sticky : 100% en BoS/Dilemme/Chicken, 0% ailleurs ; scot : 100% en BoS/Chicken (verrouillé à `CC` par sa prédiction mutuelle), 0% ailleurs ; BR : 0% partout par construction de la mesure. Le pattern « le modèle sait mais n'agit pas » est ici une propriété de *modèles déclarés*, confrontés à une prédiction de référence — plus jamais d'une constante comparée à une best response.


## 8. Exercices

Cette section rassemble **3 exercices progressifs** sur la dissociation prédire/agir.

### Exercice 1 — Remplacer le joueur simulé par un vrai LLM (openai-compatible)

L'objectif : remplacer `sticky_preferred` par un appel provider réel. Quand `OPENAI_API_KEY` (ou équivalent) est présent dans `os.environ`, le notebook appelle le modèle ; sinon, il reste sur le stub `sticky_preferred`. C'est la cellule-type d'extension **RECOVERABLE-LOCAL** (cf [sota-not-workaround.md](../../.claude/rules/sota-not-workaround.md)).

### Exercice 2 — Caractériser les swaps qui augmentent la dissociation

L'objectif : pour chaque chambre X, lister **tous** les swaps qui transforment le Nash et mesurer la dissociation sticky vs BR sur chaque swap. Identifier les swaps qui **cachent** la dissociation (BattleSexes+C12) vs ceux qui la **révèlent** (Dilemme+C23).

### Exercice 3 — Cyclicité de BattleSexes : formalisation en logique modale

L'objectif : proposer un encodage **non-transitif** de BattleSexes qui préserve ses deux Nash (C,D) et (D,C). Indice : utiliser des préférences **lexicographiques** ou une **logique modale KD45**.

In [10]:
# Exercice 1 : Remplacer le joueur simule par un vrai LLM (openai-compatible)
def call_llm_provider(history: List[Tuple[str, str]], player: str,
                      api_key: str = None) -> str:
    """
    Appelle un modele openai-compatible (OpenAI, OpenRouter, Anthropic via gateway).
    Si pas de cle API, retourne un stub C.1 (None) -- le notebook reste executable.

    Indice etudiant :
      - Utiliser `os.environ.get("OPENAI_API_KEY")` ou equivalent
      - Prompt : convertir l'historique en regles textuelles (F/J) comme dans le papier
      - Temperature 0, max_tokens 1 (mono-token)
      - Si le provider repond, retourner 'C' ou 'D' ; sinon retourner None (stub)
    """
    api_key = api_key or os.environ.get("OPENAI_API_KEY")
    if not api_key:
        # Pas de provider : stub C.1
        return None
    # TODO etudiant : implementer l'appel provider
    # Indice : openai.OpenAI(api_key=...).chat.completions.create(...)
    # Exemple : client = openai.OpenAI(api_key=api_key)
    #           resp = client.chat.completions.create(
    #               model="gpt-4",
    #               messages=[{"role": "user", "content": build_prompt(history, player)}],
    #               temperature=0, max_tokens=1)
    #           return parse_action(resp.choices[1.message.content)
    return None  # Stub C.1


# Indice : pour la construction du prompt, transformer l'historique en texte :
def build_prompt_template(history: List[Tuple[str, str]], player: str, game_name: str) -> str:
    """Template du prompt envoye au modele. Les regles textuelles F/J evitement le biais semantique."""
    opponent = "Col" if player == "Row" else "Row"
    rows = [
        f"You are playing a repeated {game_name} game.",
        f"On each round, you choose F or J. The other player ({opponent}) also chooses.",
        f"History (most recent last):",
    ]
    for i, (r, c) in enumerate(history):
        rows.append(f"  Round {i+1}: F" if (r == "C" if player == "Row" else c == "C") else f"  Round {i+1}: J")
    rows.append("Choose F or J for the next round. Reply with one character only.")
    return "\n".join(rows)


# Stub : si pas de cle, on retourne None et le notebook reste executable
print("=== Exercice 1 : call_llm_provider ===")
result = call_llm_provider([("C", "C"), ("C", "C")], "Row")
print(f"Sans cle API, retourne : {result}")
print("=> Pour utiliser un vrai LLM : configurer OPENAI_API_KEY dans .env ou os.environ")

=== Exercice 1 : call_llm_provider ===
Sans cle API, retourne : None
=> Pour utiliser un vrai LLM : configurer OPENAI_API_KEY dans .env ou os.environ


In [11]:
# Exercice 2 : Caracteriser les swaps qui augmenent la dissociation
def dissociation_post_swap(g: OrdinalGame, swap: str,
                          n_total: int = 20, swap_round: int = 10) -> Tuple[float, float]:
    """
    Mesure la dissociation sticky vs BR apres le swap.
    Retourne (dissociation_sticky_post, dissociation_BR_post) en pourcentage.
    """
    h_sticky = play_with_swap(g, swap, swap_round=swap_round, n_total=n_total, mode="sticky_preferred")
    h_br = play_with_swap(g, swap, swap_round=swap_round, n_total=n_total, mode="best_response")
    d_sticky = dissociation_rate(g, h_sticky[swap_round:])  # post-swap seulement
    d_br = dissociation_rate(g, h_br[swap_round:])
    return d_sticky, d_br


# Indice etudiant : pour chaque chambre X, iterer sur tous les swaps R{i}{j} et C{i}{j}
# valides (0 <= i < j <= 3), mesurer dissociation_post_swap(X, swap), et retourner
# les swaps qui maximisent la dissociation sticky vs BR.
def find_max_dissociation_swap(g: OrdinalGame) -> List[Tuple[str, float, float]]:
    """
    Pour la chambre g, retourne les swaps (swap, d_sticky, d_br) tries par
    dissociation sticky decroissante.
    """
    # TODO etudiant : iterer sur tous les swaps valides (6 R-swaps + 6 C-swaps),
    # appeler dissociation_post_swap, et retourner la liste triee.
    return []  # Stub C.1


# Demonstration partielle : sur Dilemme
print("=== Exercice 2 : swaps qui revelent la dissociation sur Dilemme ===")
results_dilemme = []
for i in range(4):
    for j in range(i+1, 4):
        for kind in ["R", "C"]:
            swap = f"{kind}{i}{j}"
            d_s, d_b = dissociation_post_swap(g=CLASSIC_GAMES["Dilemme"], swap=swap)
            results_dilemme.append((swap, d_s, d_b))
# Trier par dissociation sticky decroissante
results_dilemme.sort(key=lambda x: -x[1])
print(f"{'Swap':6s} {'d_sticky_post':15s} {'d_BR_post':15s}")
for swap, d_s, d_b in results_dilemme[:6]:
    print(f"{swap:6s} {d_s:>13.0%}  {d_b:>13.0%}")

=== Exercice 2 : swaps qui revelent la dissociation sur Dilemme ===
Swap   d_sticky_post   d_BR_post      
R01             100%            50%
C01             100%             0%
R02             100%             0%
C02             100%            50%
R03             100%             0%
C03             100%             0%


In [12]:
# Exercice 3 : Cyclicite de BattleSexes -- formalisation non-transitive
# Indice : pour representer BoS canonique avec 2 Nash (C,D) et (D,C), il faut
# autoriser des preferences NON transitives (le joueur peut preferer C a D,
# D a (C,C), et (C,C) a C, etc.). Une solution : utiliser des "circles de
# preference" plutot que des rangs lineaires.

from typing import Dict, Set  # noqa: E402  (import local pour la cellule exercice)

def best_response_nontransitive(g_cyclic: Dict[Tuple[str, str], Set[str]],
                                player: str, opponent_action: str) -> str:
    """
    Pour une representation cyclique des preferences :
    g_cyclic[action] = ensemble des actions strictement preferees.
    Retourne la meilleure reponse selon cette relation cyclique.

    Indice etudiant :
      - Si g_cyclic[(C,C)] contient D, alors C < D quand (C,C) est joue
      - Si g_cyclic[(D,C)] contient C, alors D < C quand (D,C) est joue
      - Pour BattleSexes : definir les 4 ensembles cycliques
    """
    # TODO etudiant : definir les 4 ensembles cycliques pour BattleSexes
    # et implementer la selection d'action
    return "C"  # Stub C.1


# Demonstration : pour BattleSexes canonique, une representation cyclique
# pourrait etre :
# - En (C,C) : Row prefere C (relation C < D, i.e. D prefere)
# - En (C,D) : Row prefere D (relation C > D, i.e. C prefere encore)
# - En (D,C) : Row prefere C (relation D < C)
# - En (D,D) : Row prefere C (relation D > C)
# Cette relation est CYCLIQUE : C < D < C, violant la transitivite.
print("=== Exercice 3 : cyclicite de BattleSexes ===")
print("Representation cyclique possible :")
print("  (C,C) : Row prefere C | (C,D) : Row prefere D")
print("  (D,C) : Row prefere C | (D,D) : Row prefere C")
print()
print("Cycle : C < D (en CC) -> D < C (en CD) -> C < D (en DC) -> D < C (en DD)")
print("=> Non representable en ordinal strict transitif.")
print("=> Stub : completer best_response_nontransitive avec les 4 ensembles.")

=== Exercice 3 : cyclicite de BattleSexes ===
Representation cyclique possible :
  (C,C) : Row prefere C | (C,D) : Row prefere D
  (D,C) : Row prefere C | (D,D) : Row prefere C

Cycle : C < D (en CC) -> D < C (en CD) -> C < D (en DC) -> D < C (en DD)
=> Non representable en ordinal strict transitif.
=> Stub : completer best_response_nontransitive avec les 4 ensembles.


## 9. Conclusion

Chaque modèle de joueur performe **structurellement** différemment selon la chambre Robinson-Goforth — et la comparaison n'a de sens qu'entre modèles déclarés :

- **BattleSexes, Dilemme, Chicken** : dissociation **maximale pour le sticky** (100% en E4) — il colle à sa première action et n'atteint pas le Nash. La BR suit les Nash déplacés (90-100% post-swap), mais n'atteint **jamais** les Nash asymétriques en jeu simultané (oscillation `CC ↔ DD`) : seul le bruit les atteint.

- **StagHunt, Harmony, Coordination** : dissociation **nulle pour le sticky** — son ancrage sur C coïncide avec un Nash. Il performe « bien » par accident, ce qu'un changement de `first_action` (Dilemme : 0% → 100%) ou un swap (Coordination+R01 : 100% → 0%) révèle immédiatement.

- **SCoT** : verdict mesuré **+0%** contre BR sur les six chambres — le gain rapporté par le papier n'est pas un théorème de la structure ordinale ; il reste une propriété empirique des modèles, tranchable par l'exercice 1.

L'apport **conceptuel** de ce notebook : la dissociation (b) du papier est re-dérivée comme propriété de **modèles de joueur** (ancrage, prédiction mutuelle, bruit) confrontés à une prédiction de référence — mesurable, paramétrable, reproductible. Et la grammaire R-G (swaps en cours de partie) **révèle** la dissociation quand elle est accidentellement cachée.

**Substrat EPITA** (#12254) : les personas de `2025-Epita-Intelligence-Symbolique` permettent l'extension directe : l'agent qui anticipe le contre-argument le réfute-t-il ? Plusieurs agents en désaccord sur la méta-action ? Ces questions héritent la grammaire de dissociation et la mesure explicite.

## Sources

- Mei et al., *Playing Repeated Games with Large Language Models*, Nature Human Behaviour (2025), [s41562-025-02172-y](https://www.nature.com/articles/s41562-025-02172-y) — page lue 2026-08-22.
- Robinson & Goforth, *The Topology of the 2x2 Games* (2005) — implémentation dans `GameTheory-3` cellule 5 (`OrdinalGame`).
- Bruns, *Austausch und Gerechtigkeit* (1975) — notion de transmutation (information nouvelle vs déplacement), voir aussi GameTheory-21 (Loi III, transformations vs morphismes).
- GameTheory-3 (chambres R-G) : [GameTheory-3-Topology2x2.ipynb](GameTheory-3-Topology2x2.ipynb)
- GameTheory-21 (morphisme fini, swaps préservants) : [GameTheory-21-Deux-Especes-de-Fleches.ipynb](GameTheory-21-Deux-Especes-de-Fleches.ipynb)

***

**Navigation** : [GameTheory-3](GameTheory-3-Topology2x2.ipynb) · **GameTheory-3c-Le-Joueur-LLM** · [GameTheory-21](GameTheory-21-Deux-Especes-de-Fleches.ipynb)
